## 1. Installation

In [7]:
!pip install -q docling
from importlib.metadata import version
print("docling version :", version("docling"))

docling version : 2.108.0


## 2. Verifier le dataset Kaggle

Sur Kaggle, les fichiers d'un dataset ajoute au notebook sont montes en lecture seule sous `/kaggle/input/<nom-du-dataset>/`. On liste le contenu pour confirmer le chemin exact avant de continuer (utile si le PDF est dans un sous-dossier).

In [8]:
import os

# Etape 1 : lister tous les datasets disponibles sous /kaggle/input
print("Datasets disponibles dans /kaggle/input :")
for name in os.listdir("/kaggle/input"):
    print(" -", name)

Datasets disponibles dans /kaggle/input :
 - datasets


In [9]:
# Etape 2 : exploration recursive complete pour trouver tous les PDF,
# quelle que soit la profondeur des sous-dossiers
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.lower().endswith(".pdf"):
            print(os.path.join(root, f))

/kaggle/input/datasets/hasnaasaadi/my-dataset/loan_0001.pdf


## 3. Configuration des parametres de lecture

Les options principales se reglent via `PdfPipelineOptions`.

In [10]:
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

pipeline_options = PdfPipelineOptions()

# --- OCR ---
# CORRECTION : do_ocr = False car les PDF sont des PDF texte natifs (non scannes).
# Docling lit directement le texte deja encode dans le PDF -> plus rapide, plus fiable.
# Mettre True uniquement si le PDF est un scan (image pure, sans texte selectionnable).
pipeline_options.do_ocr = False

# --- Tableaux ---
pipeline_options.do_table_structure = True     # reconstruit la structure des tableaux
                                               # (essentiel pour les formulaires de credit)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)
print("Converter pret.")

Converter pret.


## 4. Lecture du PDF

Chemin adapte a Kaggle : `/kaggle/input/<dataset>/...` pour la lecture (lecture seule).

In [14]:
# Adapte le nom de fichier si besoin, d'apres la sortie de la cellule 2
DATASET_DIR = "/kaggle/input/datasets/hasnaasaadi/my-dataset"
PDF_PATH = os.path.join(DATASET_DIR, "loan_0001.pdf")

result = converter.convert(PDF_PATH)

doc = result.document
print("Statut :", result.status)
print("Nombre de pages :", len(doc.pages))

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Statut : ConversionStatus.SUCCESS
Nombre de pages : 2


## 5. Export du contenu

Le Markdown est le plus lisible, le dict/JSON est le plus complet pour un traitement automatique en aval.

In [15]:
md_text = doc.export_to_markdown()
print(md_text)

## Loan Application for a Commercial Real Estate Loan - Company Profile

## 1. Applicant

| Company Name                       | Söding GmbH & Co. KGaA UG                   |
|------------------------------------|---------------------------------------------|
| Legal Form                         | Limited Partnership (KG)                    |
| Date of Incorporation              | 01/12/2006                                  |
| Business Address                   | Falk-Textor-Weg 160, 86608 Gransee, Germany |
| Commercial Register Number / Court | HRB 977141 / Gransee Local Court            |
| VAT ID / Tax Number                | DE681242882                                 |

## 2. Financial Request / Property Description

| Type of Property                    | Office and Commercial Building                 |
|-------------------------------------|------------------------------------------------|
| Property Name                       | Werden Center Gransee                          |

In [16]:
# Sauvegarde en Markdown dans /kaggle/working (seul dossier ecriture autorisee sur Kaggle)
OUTPUT_DIR = "/kaggle/working"
md_out_path = os.path.join(OUTPUT_DIR, "loan_0001.md")

with open(md_out_path, "w", encoding="utf-8") as f:
    f.write(md_text)

print("Sauvegarde OK :", md_out_path)

Sauvegarde OK : /kaggle/working/loan_0001.md


In [17]:
# Dictionnaire complet (structure + positions), utile pour extraire des champs automatiquement
import json

data = doc.export_to_dict()
print("Cles principales :", list(data.keys()))

json_out_path = os.path.join(OUTPUT_DIR, "loan_0001_docling.json")
with open(json_out_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)
print("JSON sauvegarde :", json_out_path)

Cles principales : ['schema_name', 'version', 'name', 'origin', 'furniture', 'body', 'groups', 'texts', 'pictures', 'tables', 'key_value_items', 'form_items', 'pages']
JSON sauvegarde : /kaggle/working/loan_0001_docling.json


## 6. Parcourir les elements avec leurs positions (bounding boxes)

Chaque element a un label (texte, titre, tableau...) et des coordonnees sur la page. Utile pour relier une valeur extraite a son emplacement physique.

In [18]:
for item, level in doc.iterate_items():
    label = getattr(item, "label", None)
    text = getattr(item, "text", "")
    prov = getattr(item, "prov", None)
    if text:
        bbox = prov[0].bbox if prov else None
        page = prov[0].page_no if prov else None
        print(f"[{label}] page {page} | bbox {bbox}")
        print("   ", text[:90])

[section_header] page 1 | bbox l=57.02362 t=779.2260705078124 r=501.58362 b=736.7260705078124 coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>
    Loan Application for a Commercial Real Estate Loan - Company Profile
[section_header] page 1 | bbox l=57.02362 t=707.4820705078124 r=125.03961999999999 b=696.3820705078124 coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>
    1. Applicant
[section_header] page 1 | bbox l=57.02362 t=537.4820705078124 r=301.76362 b=526.3820705078124 coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>
    2. Financial Request / Property Description
[section_header] page 1 | bbox l=57.02362 t=301.4820705078125 r=181.72762 b=290.3820705078125 coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>
    3. Financing Proposal
[section_header] page 1 | bbox l=57.02362 t=131.48207050781252 r=206.39961999999997 b=120.3820705078125 coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>
    4. Declaration & Signature
[text] page 1 | bbox l=57.02362 t=108.32811050781243 r=3

## 7. Recuperer les tableaux

Le formulaire de credit est surtout compose de tableaux (etiquette / valeur). Docling les extrait directement, exportables en DataFrame pandas.

In [19]:
import pandas as pd

tables = doc.tables
print("Nombre de tableaux detectes :", len(tables))

for i, table in enumerate(tables):
    df = table.export_to_dataframe()
    print(f"\n--- Tableau {i+1} ---")
    print(df)

Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


Nombre de tableaux detectes : 3

--- Tableau 1 ---
                         Company Name  \
0                          Legal Form   
1               Date of Incorporation   
2                    Business Address   
3  Commercial Register Number / Court   
4                 VAT ID / Tax Number   

                     Söding GmbH & Co. KGaA UG  
0                     Limited Partnership (KG)  
1                                   01/12/2006  
2  Falk-Textor-Weg 160, 86608 Gransee, Germany  
3             HRB 977141 / Gransee Local Court  
4                                  DE681242882  

--- Tableau 2 ---
                                     0  \
0                     Type of Property   
1                        Property Name   
2                               Adress   
3  Purchase Price / Construction Costs   
4             Desired Financing Amount   
5                       Purpose of Use   
6                  Equity Contribution   
7                 Year of Construction   
8          

In [20]:
with open("/kaggle/working/loan_0001.md", "r", encoding="utf-8") as f:
    print(f.read())

## Loan Application for a Commercial Real Estate Loan - Company Profile

## 1. Applicant

| Company Name                       | Söding GmbH & Co. KGaA UG                   |
|------------------------------------|---------------------------------------------|
| Legal Form                         | Limited Partnership (KG)                    |
| Date of Incorporation              | 01/12/2006                                  |
| Business Address                   | Falk-Textor-Weg 160, 86608 Gransee, Germany |
| Commercial Register Number / Court | HRB 977141 / Gransee Local Court            |
| VAT ID / Tax Number                | DE681242882                                 |

## 2. Financial Request / Property Description

| Type of Property                    | Office and Commercial Building                 |
|-------------------------------------|------------------------------------------------|
| Property Name                       | Werden Center Gransee                          |

## Resume des chemins Kaggle

- Lecture (input, lecture seule) : `/kaggle/input/loan-pdfs/...`
- Ecriture (output) : `/kaggle/working/...` -> c'est ce dossier qui apparait ensuite dans l'onglet "Output" du notebook Kaggle une fois execute/commit

## Resume des parametres principaux

- `do_ocr = False` : PDF texte natif, pas de scan -> lecture directe, plus rapide (CORRECTION cle)
- `do_table_structure = True` : reconstruire les tableaux
- Sorties utiles : `export_to_markdown()`, `export_to_dict()`, `doc.tables`, `doc.iterate_items()`